In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import nibabel as nib
import numpy as np
import shutil
from pathlib import Path

MASK_PATH = Path("/content/drive/MyDrive/FINAL_DATABASE/ADNI_DATABASE/DementiaMask_AAL3.nii")
SRC   = Path("/content/drive/MyDrive/FINAL_DATABASE/ADNI_DATABASE/PROCESSED/ADNI_MRI_SPM_STANDARDISED/normalised_mri")
DST   = Path("/content/drive/MyDrive/FINAL_DATABASE/ADNI_DATABASE/PROCESSED/ADNI_MRI_SPM_STANDARDISED/masked_normalised_mri")
LOCAL = Path("/content/work_mask/mri")

DST.mkdir(parents=True, exist_ok=True)
LOCAL.mkdir(parents=True, exist_ok=True)

# Maskeyi bir kez yükle
mask = nib.load(str(MASK_PATH)).get_fdata().astype(np.float32)
print(f"Maske shape: {mask.shape}  ← (91,109,91) olmalı")

files = list(SRC.glob("*.nii"))
print(f"MRI toplam : {len(files)}")

done = skipped = error = 0
for i, f in enumerate(files):
    try:
        dst = DST / f.name
        if dst.exists():
            skipped += 1
            continue

        img  = nib.load(str(f))
        data = img.get_fdata(dtype=np.float32)  # (91,109,91)

        # Maskele
        masked = (data * mask).astype(np.float32)

        local_f = LOCAL / f.name
        nib.save(nib.Nifti1Image(masked, img.affine), str(local_f))
        shutil.move(str(local_f), dst)
        done += 1

        if (i+1) % 200 == 0:
            print(f"  {i+1}/{len(files)} — done:{done} skipped:{skipped}")

    except Exception as e:
        error += 1
        print(f"  HATA: {f.name} → {e}")

print(f"\n✅ MRI maskeleme bitti!")
print(f"  Done   : {done}")
print(f"  Skipped: {skipped}")
print(f"  Error  : {error}")
print(f"  DST    : {len(list(DST.glob('*.nii')))} / {len(files)}")

Maske shape: (91, 109, 91)  ← (91,109,91) olmalı
MRI toplam : 3978
  200/3978 — done:200 skipped:0
  400/3978 — done:400 skipped:0
  600/3978 — done:600 skipped:0
  800/3978 — done:800 skipped:0
  1000/3978 — done:1000 skipped:0
  1200/3978 — done:1200 skipped:0
  1400/3978 — done:1400 skipped:0
  1600/3978 — done:1600 skipped:0
  1800/3978 — done:1800 skipped:0
  2000/3978 — done:2000 skipped:0
  2200/3978 — done:2200 skipped:0
  2400/3978 — done:2400 skipped:0
  2600/3978 — done:2600 skipped:0
  2800/3978 — done:2800 skipped:0
  3000/3978 — done:3000 skipped:0
  3200/3978 — done:3200 skipped:0
  3400/3978 — done:3400 skipped:0
  3600/3978 — done:3600 skipped:0
  3800/3978 — done:3800 skipped:0

✅ MRI maskeleme bitti!
  Done   : 3978
  Skipped: 0
  Error  : 0
  DST    : 3978 / 3978
